In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
df = pd.read_csv("Data/dialogues.tsv", sep="\t")

In [3]:
df = df["dialogue"]

In [4]:
texts = df[45]

## Предосмотр диалога

In [5]:
import textwrap

wrapped_text = textwrap.fill(texts, width=50)
print(wrapped_text)

<span class=participant_2>Пользователь 2:
Привет</span><br /><span
class=participant_1>Пользователь 1:
привет</span><br /><span
class=participant_2>Пользователь 2: Как дела
?</span><br /><span
class=participant_1>Пользователь 1:
отлично</span><br /><span
class=participant_2>Пользователь 2: Чем
занимаешься ?</span><br /><span
class=participant_1>Пользователь 1: чеем
занимаешься</span><br /><span
class=participant_1>Пользователь 1: недавно с
басейна пришёл отдыхаю</span><br /><span
class=participant_2>Пользователь 2: Я
работаю</span><br /><span
class=participant_2>Пользователь 2: Давно в
бассейн ходишь ?</span><br /><span
class=participant_1>Пользователь 1: да обожаю
воду</span><br /><span
class=participant_1>Пользователь 1: и ещё
машины</span><br /><span
class=participant_2>Пользователь 2: А я наоборот
не очень</span><br /><span
class=participant_2>Пользователь 2: Как проводишь
свободное время ?</span><br /><span
class=participant_1>Пользователь 1: а ты чем
занимаешься?вообще</span><br 

## Обрезание тегов

In [25]:
pattern = r"participant_(\d)>Пользователь \d:(.*?)(?=</span>)"

matches = re.findall(pattern=pattern, string=texts)

In [26]:
matches

[('2', ' Привет'),
 ('1', ' привет'),
 ('2', ' Как дела ?'),
 ('1', ' отлично'),
 ('2', ' Чем занимаешься ?'),
 ('1', ' чеем занимаешься'),
 ('1', ' недавно с басейна пришёл отдыхаю'),
 ('2', ' Я работаю'),
 ('2', ' Давно в бассейн ходишь ?'),
 ('1', ' да обожаю воду'),
 ('1', ' и ещё машины'),
 ('2', ' А я наоборот не очень'),
 ('2', ' Как проводишь свободное время ?'),
 ('1', ' а ты чем занимаешься?вообще'),
 ('2', ' Люблю рыбалку'),
 ('1', ' нет свободного времени вообще'),
 ('2', ' У тебя есть жена , дети ?'),
 ('1', ' да есть'),
 ('1', ' 3 детей'),
 ('2', ' Работаешь ?'),
 ('1', ' да вахтой'),
 ('2', ' Далеко ездишь ?'),
 ('1', ' 1004 км'),
 ('2', ' Из какого ты города ?'),
 ('1', ' сыктывкар'),
 ('1', ' а ты?'),
 ('2', ' Я с Украины'),
 ('2', ' Есть домашние животные?'),
 ('1', ' у вс наверно уже весна'),
 ('2', ' Ещё нет (( снег идёт'),
 ('1', ' кошка кот собака(алабай)'),
 ('2', ' У меня только кот'),
 ('2', ' Вы живёте в частном доме ?'),
 ('1', ' да в частном'),
 ('2', ' Или 

## Восстановление структуры диалога

In [8]:
dialog = []
temp = matches[0][1]
temp_number = matches[0][0]

for i in range(1, len(matches)):
    if matches[i][0] != temp_number:
        if(len(temp) > 0):
            dialog.append((temp_number, temp))

        temp_number = matches[i][0]
        temp = matches[i][1]

    else:
        temp += " " + matches[i][1]

if len(temp) > 0: dialog.append((temp_number, temp))
dialog


[('2', ' Привет'),
 ('1', ' привет'),
 ('2', ' Как дела ?'),
 ('1', ' отлично'),
 ('2', ' Чем занимаешься ?'),
 ('1', ' чеем занимаешься  недавно с басейна пришёл отдыхаю'),
 ('2', ' Я работаю  Давно в бассейн ходишь ?'),
 ('1', ' да обожаю воду  и ещё машины'),
 ('2', ' А я наоборот не очень  Как проводишь свободное время ?'),
 ('1', ' а ты чем занимаешься?вообще'),
 ('2', ' Люблю рыбалку'),
 ('1', ' нет свободного времени вообще'),
 ('2', ' У тебя есть жена , дети ?'),
 ('1', ' да есть  3 детей'),
 ('2', ' Работаешь ?'),
 ('1', ' да вахтой'),
 ('2', ' Далеко ездишь ?'),
 ('1', ' 1004 км'),
 ('2', ' Из какого ты города ?'),
 ('1', ' сыктывкар  а ты?'),
 ('2', ' Я с Украины  Есть домашние животные?'),
 ('1', ' у вс наверно уже весна'),
 ('2', ' Ещё нет (( снег идёт'),
 ('1', ' кошка кот собака(алабай)'),
 ('2', ' У меня только кот  Вы живёте в частном доме ?'),
 ('1', ' да в частном'),
 ('2', ' Или в квартире ?  Как тебя зовут ?'),
 ('1', ' кот породистый?'),
 ('2', ' Обычный ) рыжий')

## Удаление стоп-слов, удаление пунктуации и пробелов, токенизация

In [36]:
corr_dialog = []
stop_words = {
    'в', 'на', 'с', 'у', 'из', 'к', 'по', 'о', 'до', 'для',
    'и', 'а', 'но', 'или', 'как', 'что', 'чтобы', 'да', 
    'не', 'ни', 'же', 'то', 'только', 'ещё', 'еже', 'уже', 'нет',
    'я', 'ты', 'вы', 'он', 'мы', 'мне', 'меня', 'тебе', 'тебя', 'тебе',
    'у вас', 'вас', 'у тебя', 'какое', 'какого', 'какой',
    'вообще', 'наоборот', 'очень'
}

for number, text in dialog:
    text = text.lower()
    tokens = re.findall(r"[a-яё0-9]+", text)
    tokens_clean = [token for token in tokens if token not in stop_words]
    
    if len(tokens_clean) > 0:
        corr_dialog.append((number, tokens_clean))

In [34]:
corr_dialog

[('2', ['привет']),
 ('1', ['привет']),
 ('2', ['дела']),
 ('1', ['отлично']),
 ('2', ['чем', 'занимаешься']),
 ('1', ['чеем', 'занимаешься', 'недавно', 'басейна', 'пришёл', 'отдыхаю']),
 ('2', ['работаю', 'давно', 'бассейн', 'ходишь']),
 ('1', ['обожаю', 'воду', 'машины']),
 ('2', ['проводишь', 'свободное', 'время']),
 ('1', ['чем', 'занимаешься']),
 ('2', ['люблю', 'рыбалку']),
 ('1', ['свободного', 'времени']),
 ('2', ['есть', 'жена', 'дети']),
 ('1', ['есть', '3', 'детей']),
 ('2', ['работаешь']),
 ('1', ['вахтой']),
 ('2', ['далеко', 'ездишь']),
 ('1', ['1004', 'км']),
 ('2', ['города']),
 ('1', ['сыктывкар']),
 ('2', ['украины', 'есть', 'домашние', 'животные']),
 ('1', ['вс', 'наверно', 'весна']),
 ('2', ['снег', 'идёт']),
 ('1', ['кошка', 'кот', 'собака', 'алабай']),
 ('2', ['кот', 'живёте', 'частном', 'доме']),
 ('1', ['частном']),
 ('2', ['квартире', 'зовут']),
 ('1', ['кот', 'породистый']),
 ('2', ['обычный', 'рыжий']),
 ('1', ['александр']),
 ('2', ['сколько', 'лет']),
 ('1'

## Лемматизация

In [ ]:
import pymorphy3

morph = pymorphy3.MorphAnalyzer()
lemmatized_dialog = []

for number, tokens in corr_dialog:
    lemmatized_tokens = []
    for token in tokens:
        lem_token = morph.parse(token)[0].normal_form
        lemmatized_tokens.append(lem_token)

    lemmatized_dialog.append((number, lemmatized_tokens))

lemmatized_dialog

[('2', ['привет']),
 ('1', ['привет']),
 ('2', ['дело']),
 ('1', ['отлично']),
 ('2', ['чем', 'заниматься']),
 ('1', ['чей', 'заниматься', 'недавно', 'басейн', 'прийти', 'отдыхать']),
 ('2', ['работать', 'давно', 'бассейн', 'ходить']),
 ('1', ['обожать', 'вода', 'машина']),
 ('2', ['проводить', 'свободный', 'время']),
 ('1', ['чем', 'заниматься']),
 ('2', ['любить', 'рыбалка']),
 ('1', ['свободный', 'время']),
 ('2', ['есть', 'жена', 'ребёнок']),
 ('1', ['есть', '3', 'ребёнок']),
 ('2', ['работать']),
 ('1', ['вахта']),
 ('2', ['далеко', 'ездить']),
 ('1', ['1004', 'км']),
 ('2', ['город']),
 ('1', ['сыктывкар']),
 ('2', ['украина', 'есть', 'домашний', 'животное']),
 ('1', ['вс', 'наверно', 'весна']),
 ('2', ['снег', 'идти']),
 ('1', ['кошка', 'кот', 'собака', 'алабай']),
 ('2', ['кот', 'жить', 'частный', 'дом']),
 ('1', ['частный']),
 ('2', ['квартира', 'звать']),
 ('1', ['кот', 'породистый']),
 ('2', ['обычный', 'рыжий']),
 ('1', ['александр']),
 ('2', ['сколько', 'год']),
 ('1', ['1

## TF-IDF

In [ ]:
tokens_set = list(set([token for num, tokens in lemmatized_dialog for token in tokens]))
IDF = {}

for uniq_token in tokens_set:
    frequently = sum([1 for num, tokens in lemmatized_dialog if uniq_token in tokens])
    idf_token = len(lemmatized_dialog)/frequently
    IDF[uniq_token] = np.log1p(idf_token)

tf_idf_dialog = []

for num, tokens in  lemmatized_dialog:
    vector = [0.0] * len(tokens_set)
    for i, token in enumerate(tokens_set):
        tf = tokens.count(token)/ len(tokens)
        vector[i] = (tf * IDF[token])

    tf_idf_dialog.append((num, vector))

tf_idf_dialog

[('2',
  [np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(3.068052935133617),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),
   np.float64(0.0),

### 10 самых важных

In [13]:
vectors_only = [vector for num, vector in tf_idf_dialog]
df_tfidf = pd.DataFrame(vectors_only, columns=tokens_set)

top_words = df_tfidf.mean(axis=0).sort_values(ascending=False)

print("--- ТОП-10 ВАЖНЫХ СЛОВ ДИАЛОГА (по версии TF-IDF) ---")
print(top_words.head(10))

--- ТОП-10 ВАЖНЫХ СЛОВ ДИАЛОГА (по версии TF-IDF) ---
привет       0.149661
частный      0.093538
работать     0.093538
сыктывкар    0.091163
вахта        0.091163
александр    0.091163
дело         0.091163
взаимно      0.091163
19           0.091163
отлично      0.091163
dtype: float64


## Эмбендинги

In [14]:
from sentence_transformers import SentenceTransformer   

In [15]:
phrases = [" ".join(tokens) for num, tokens in lemmatized_dialog]
numbers = [num for num, tokens in lemmatized_dialog]
phrases

['привет',
 'привет',
 'дело',
 'отлично',
 'чем заниматься',
 'чей заниматься недавно басейн прийти отдыхать',
 'работать давно бассейн ходить',
 'обожать вода машина',
 'проводить свободный время',
 'чем заниматься',
 'любить рыбалка',
 'свободный время',
 'есть жена ребёнок',
 'есть 3 ребёнок',
 'работать',
 'вахта',
 'далеко ездить',
 '1004 км',
 'город',
 'сыктывкар',
 'украина есть домашний животное',
 'вс наверно весна',
 'снег идти',
 'кошка кот собака алабай',
 'кот жить частный дом',
 'частный',
 'квартира звать',
 'кот породистый',
 'обычный рыжий',
 'александр',
 'сколько год',
 '19',
 'трое ребёнок',
 'зимой рыбачить',
 'зимой рыбачить обожать рыба',
 'два сыночек лапочка дочка погода сейчас рыбачить позволять',
 'холодно хобби',
 'погодный качели 20 5 бассейн машина',
 'круто быть приятно познакомиться',
 'взаимно',
 'выращивать цитрусовый хороший день']

In [16]:
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
embeddings = model.encode(phrases)
np_numbers = np.array(numbers).reshape(-1, 1)

embedding_dialog = np.concatenate([np_numbers, embeddings], axis=1)
embedding_dialog


array([['2', '0.16597949', '0.15848604', ..., '0.20053379',
        '-0.14684321', '-0.041775003'],
       ['1', '0.16597949', '0.15848604', ..., '0.20053379',
        '-0.14684321', '-0.041775003'],
       ['2', '-0.0047473945', '0.14295626', ..., '-0.080173776',
        '0.03596561', '0.25616178'],
       ...,
       ['2', '0.3398012', '0.12866917', ..., '0.25864568', '-0.21023671',
        '0.075886905'],
       ['1', '0.19261873', '0.04082669', ..., '-0.13231651',
        '0.08644735', '0.009354721'],
       ['2', '-0.13622421', '0.07602314', ..., '0.21551915',
        '0.22243203', '0.27742854']], dtype='<U32')